In [0]:
use catalog identifier(:catalog)

### Q01 — Overall KPI Summary: Net Revenue, Orders, Active Customers, Average Order Value

In [0]:

SELECT
  ROUND(SUM(net_revenue), 2)                                        AS net_revenue,
  COUNT(DISTINCT l_orderkey)                                        AS orders,
  COUNT(DISTINCT o_custkey)                                         AS active_customers,
  ROUND(SUM(net_revenue) / COUNT(DISTINCT l_orderkey), 2)           AS avg_order_value
FROM gold.fact_sales;

**Interpretation:** Across all years the business generated **$1,089.84B net revenue** from **7,500,000 orders** and **499,989 active customers** (66.7% of the 750,000-customer base), with an average order value of **$145,311.36**.

**Management action:** Track AOV and active-customer share as the two headline KPIs; the roughly one-third dormant customers are the largest immediate growth lever.

### Q02 — Monthly Revenue Trend Across All Years

In [0]:
SELECT
  YEAR(o_orderdate)                       AS order_year,
  MONTH(o_orderdate)                      AS order_month,
  ROUND(SUM(net_revenue), 2)              AS net_revenue
FROM gold.fact_sales
GROUP BY order_year, order_month
ORDER BY order_year, order_month;

**Interpretation:** Monthly revenue is remarkably stable from 1992 through mid-1998, with no pronounced seasonality — each month contributes a similar share of the annual total.

**Management action:** With a flat trend and no seasonal peaks, capacity and inventory should be planned at a steady run-rate; growth must come from share gains, not seasonal campaigns.

### Q03 — Annual Revenue with Year-over-Year Change

In [0]:
WITH annual AS (
  SELECT YEAR(o_orderdate) AS order_year, SUM(net_revenue) AS net_revenue
  FROM gold.fact_sales
  GROUP BY 1
)
SELECT
  order_year,
  ROUND(net_revenue, 2) AS net_revenue,
  ROUND(net_revenue - LAG(net_revenue) OVER (ORDER BY order_year), 2) AS yoy_change,
  ROUND(100.0 * (net_revenue - LAG(net_revenue) OVER (ORDER BY order_year))
        / NULLIF(LAG(net_revenue) OVER (ORDER BY order_year), 0), 2) AS yoy_change_pct
FROM annual
ORDER BY order_year;

**Interpretation:** Annual revenue is essentially flat at **$165.0–166.0B** from 1992–1997 (YoY swings within ±0.6%). The apparent **−41.3%** drop in 1998 is a data-coverage artifact — the dataset ends on 2 August 1998, so the year is incomplete.

**Management action:** Treat 1998 as a partial year in all reporting; investigate only if the normalized run-rate also declines.

### Q04 — Revenue by Customer Market Segment

In [0]:

SELECT
  c_mktsegment,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(100.0 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS pct_of_total
FROM gold.fact_sales
GROUP BY 1
ORDER BY net_revenue DESC;

**Interpretation:** Revenue is split almost perfectly evenly across the five market segments — FURNITURE leads with **20.10%** and HOUSEHOLD trails with **19.88%**, a spread of only 0.22 percentage points.

**Management action:** Do not prioritize segments on current revenue share; differentiation decisions should instead be driven by margin (Q19) and returns (Q20).

### Q05 — Top 10 Customers by Net Revenue

In [0]:
SELECT
  o_custkey            AS customer_key,
  customer_name,
  c_mktsegment,
  customer_nation,
  ROUND(SUM(net_revenue), 2) AS net_revenue
FROM gold.fact_sales
GROUP BY 1, 2, 3, 4
ORDER BY net_revenue DESC
LIMIT 10;

**Interpretation:** The top customer (Customer#000721180, BUILDING segment, Romania) generated **$6.88M**; the top 10 span **$6.52M–$6.88M** and come from different segments and nations.

**Management action:** Treat the top 10 as a cohort of similarly sized strategic accounts and offer them a structured key-account program rather than chasing a single dominant customer.

### Q06 — Top 10 Customers as a Share of Total Revenue

In [0]:

WITH customer_rev AS (
  SELECT o_custkey, customer_name, SUM(net_revenue) AS net_revenue
  FROM gold.fact_sales
  GROUP BY 1, 2
),
total_rev AS (
  SELECT SUM(net_revenue) AS total_net_revenue FROM gold.fact_sales
)
SELECT
  c.o_custkey AS customer_key,
  c.customer_name,
  ROUND(c.net_revenue, 2) AS net_revenue,
  ROUND(100.0 * c.net_revenue / t.total_net_revenue, 4) AS pct_of_total_revenue
FROM customer_rev c
CROSS JOIN total_rev t
ORDER BY c.net_revenue DESC
LIMIT 10;

**Interpretation:** Even the largest customer contributes only **~0.0006%** of total revenue, and the entire top 10 together contribute under **0.01%** — revenue concentration is effectively zero.

**Management action:** Single-account churn poses negligible risk; prioritize broad retention and wallet-share programs over individual account rescue efforts.

### Q07 — Top 10 Products by Net Revenue

In [0]:

SELECT
  l_partkey            AS product_key,
  product_name,
  product_type,
  ROUND(SUM(net_revenue), 2) AS net_revenue
FROM gold.fact_sales
GROUP BY 1, 2, 3
ORDER BY net_revenue DESC
LIMIT 10;

**Interpretation:** The best-selling product (part 392955, SMALL BURNISHED STEEL) earned **$2.92M**; the top 10 products cluster tightly at **$2.70–2.92M**.

**Management action:** With a 1M-product catalog and tiny top-SKU contribution, this is a long-tail business — maintain broad assortment and invest in search/recommendation rather than hero-SKU marketing.

## Q08 — Bottom 10 Products by Net Revenue

In [0]:

SELECT
  l_partkey            AS product_key,
  product_name,
  product_type,
  COUNT(*)                   AS order_lines,
  ROUND(SUM(net_revenue), 2) AS net_revenue
FROM gold.fact_sales
GROUP BY 1, 2, 3
ORDER BY net_revenue ASC
LIMIT 10;

**Interpretation:** The bottom 10 products still sold — each generated **$212K–$240K** on only 8–16 order lines.

**Management action:** Before delisting, check whether low revenue reflects low price points or genuinely weak demand; the catalog floor is high enough that pruning decisions need margin data (Q19).

## Q09 — Revenue by Product Type

In [0]:
SELECT
  product_type,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(100.0 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS pct_of_total
FROM gold.fact_sales
GROUP BY 1
ORDER BY net_revenue DESC;

**Interpretation:** Across the **150 product types**, revenue is strikingly even: the top type (ECONOMY PLATED STEEL) earned **$7.58B** and the bottom (LARGE BURNISHED STEEL) **$6.99B** — a spread of under 8%.

**Management action:** No product family over- or under-indexes on top-line; assortment and sourcing decisions should be driven by supply cost and return behavior (Q19, Q20) rather than revenue mix.

## Q10 — Top 10 Suppliers by Net Revenue

In [0]:

SELECT
  l_suppkey            AS supplier_key,
  supplier_name,
  supplier_nation,
  ROUND(SUM(net_revenue), 2) AS net_revenue
FROM gold.fact_sales
GROUP BY 1, 2, 3
ORDER BY net_revenue DESC
LIMIT 10;

**Interpretation:** The largest supplier (Supplier#000007491, Saudi Arabia) booked **$29.24M**; the top 10 suppliers are tightly clustered at **$28.5–29.2M** and span many different nations.

**Management action:** The top suppliers are near-interchangeable in size, which gives procurement strong negotiating leverage and easy substitution options.

## Q11 — Revenue Share of the Top 10 Suppliers

In [0]:
WITH supplier_rev AS (
  SELECT l_suppkey, SUM(net_revenue) AS net_revenue
  FROM gold.fact_sales
  GROUP BY 1
),
ranked AS (
  SELECT s.*, ROW_NUMBER() OVER (ORDER BY s.net_revenue DESC) AS rnk
  FROM supplier_rev s
),
total_rev AS (
  SELECT SUM(net_revenue) AS total_net_revenue FROM gold.fact_sales
)
SELECT
  r.rnk,
  r.l_suppkey AS supplier_key,
  ROUND(r.net_revenue, 2) AS net_revenue,
  ROUND(100.0 * r.net_revenue / t.total_net_revenue, 4) AS pct_of_total_revenue
FROM ranked r
CROSS JOIN total_rev t
WHERE r.rnk <= 10
ORDER BY r.rnk;

**Interpretation:** The top 10 suppliers together account for **$288.9M**, only **0.0265%** of total net revenue.

**Management action:** There is no supplier spend-concentration risk; dual-sourcing decisions should be based on capability and lead time, not spend dependency.

## Q12 — Revenue by Customer Region

In [0]:

SELECT
  customer_region,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(100.0 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS pct_of_total
FROM gold.fact_sales
GROUP BY 1
ORDER BY net_revenue DESC;

**Interpretation:** All five customer regions generate almost identical revenue — EUROPE leads at **20.09%**, AFRICA trails at **19.88%**.

**Management action:** The footprint is fully balanced, so regional growth initiatives must be justified by local cost-to-serve or margin, not by revenue upside in an under-penetrated region.

## Q13 — Revenue by Customer Nation (All Nations)

In [0]:

SELECT
  customer_nation,
  customer_region,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(100.0 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS pct_of_total
FROM gold.fact_sales
GROUP BY 1, 2
ORDER BY net_revenue DESC;

**Interpretation:** All 25 nations contribute **~$43.4–44.2B** each; the top five are IRAQ ($44.20B), INDONESIA ($44.16B), GERMANY ($43.97B), FRANCE ($43.95B) and IRAN ($43.95B).

**Management action:** With revenue evenly spread, country-level strategy should focus on operational factors (logistics cost, FX, trade risk) rather than chasing volume differences.

## Q14 — Revenue and Volume Split by Return Flag

In [0]:
SELECT
  l_returnflag,
  COUNT(*)                                AS order_lines,
  SUM(l_quantity)                         AS quantity,
  ROUND(SUM(net_revenue), 2)              AS net_revenue,
  ROUND(100.0 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS pct_of_revenue
FROM gold.fact_sales
GROUP BY 1
ORDER BY 1;

**Interpretation:** About half of all lines are still in flight (**N**, 50.6%, $551.7B), while **returned lines (R)** account for **24.7%** of volume and **$269.1B** of revenue — essentially the same share as approved/delivered (**A**, 24.7%, $269.0B).

**Management action:** A ~25% return rate is the single biggest profit leak in the business; launch a cross-functional root-cause program (product accuracy, quality, handling) to reduce it.

## Q15 — Average Discount by Market Segment

In [0]:
SELECT
  c_mktsegment,
  ROUND(AVG(l_discount), 4) AS avg_discount,
  MIN(l_discount)           AS min_discount,
  MAX(l_discount)           AS max_discount
FROM gold.fact_sales
GROUP BY 1
ORDER BY avg_discount DESC;

**Interpretation:** The average discount is exactly **5.0% in every market segment** (range 0–10%), with no differentiation between segments.

**Management action:** Pricing has headroom to become strategic — differentiate discounts by segment value or loyalty instead of applying a uniform 5%.

## Q16 — Average Ship-to-Receipt Days by Shipping Mode

In [0]:
SELECT
  l_shipmode,
  COUNT(*)                                           AS order_lines,
  ROUND(AVG(DATEDIFF(l_receiptdate, l_shipdate)), 2) AS avg_ship_to_receipt_days,
  MIN(DATEDIFF(l_receiptdate, l_shipdate))           AS min_days,
  MAX(DATEDIFF(l_receiptdate, l_shipdate))           AS max_days
FROM gold.fact_sales
GROUP BY 1
ORDER BY avg_ship_to_receipt_days ASC;

**Interpretation:** All seven ship modes deliver in an average of **15.5 days** (range 1–30 days, FOB marginally slowest at 15.51), each carrying ~4.28M order lines — no mode is meaningfully faster.

**Management action:** Since speed does not differentiate modes, carrier strategy should be renegotiated on cost, reliability variance and damage rates rather than promised transit times.

## Q17 — Order Count and Revenue by Order Priority

In [0]:
SELECT
  o_orderpriority,
  COUNT(DISTINCT l_orderkey) AS orders,
  COUNT(*)                   AS order_lines,
  ROUND(SUM(net_revenue), 2) AS net_revenue,
  ROUND(100.0 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS pct_of_revenue
FROM gold.fact_sales
GROUP BY 1
ORDER BY net_revenue DESC;

**Interpretation:** Order priorities are evenly split — each class carries ~1.5M orders (~20% of volume) and ~$217.6–218.2B of revenue, with **1-URGENT** marginally leading at **$218.18B (20.02%)** and 3-MEDIUM trailing at **$217.59B (19.97%)**.

**Management action:** Urgent demand is a fifth of total volume, not a small premium slice — operations must plan capacity to serve URGENT orders as standard load, and consider premium pricing for priority handling.

## Q18 — Monthly Breakdown of Order Status

In [0]:

SELECT
  YEAR(o_orderdate)          AS order_year,
  MONTH(o_orderdate)         AS order_month,
  o_orderstatus,
  COUNT(DISTINCT l_orderkey) AS orders,
  ROUND(SUM(net_revenue), 2) AS net_revenue
FROM gold.fact_sales
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3;

**Interpretation:** The monthly status breakdown traces a single lifecycle wave: every order through **Jan 1995 is Final (F)**; **Feb–Jul 1995** is a short transition mixing F, Pending (P) and Open (O); from **Aug 1995 onward every order is Open (O)**. Monthly revenue holds at ~$13.5–14.1B throughout, so the shift reflects order age relative to the data snapshot, not performance.

**Management action:** Do not read the F→O shift as a fulfilment slowdown — measure fulfilment with ship/receipt lags (Q16) instead, and treat the Open cohort as normal in-flight backlog.

## Q19 — Product Revenue Ranking Within Each Market Segment

In [0]:

WITH product_segment_rev AS (
  SELECT
    c_mktsegment,
    l_partkey           AS product_key,
    product_name,
    product_type,
    SUM(net_revenue)    AS net_revenue
  FROM gold.fact_sales
  GROUP BY 1, 2, 3, 4
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY c_mktsegment ORDER BY net_revenue DESC) AS rnk
  FROM product_segment_rev
)
SELECT
  c_mktsegment,
  rnk,
  product_key,
  product_name,
  product_type,
  ROUND(net_revenue, 2) AS net_revenue
FROM ranked
WHERE rnk <= 5
ORDER BY c_mktsegment, rnk;

**Interpretation:** Each segment's top-5 products earn **$0.93–1.16M** — HOUSEHOLD has the strongest leader (**$1.16M**, product 327918), AUTOMOBILE next ($1.12M), FURNITURE the softest ($1.04M) — and the leaders are entirely different products in each segment.

**Management action:** No product dominates any segment, confirming the long-tail pattern from Q07/Q09 — run segment-specific assortment and promotion programs on the local top-50 rather than a single global hero list.

## Q20 — Revenue Comparison of Customer Geography vs Supplier Geography by Region

In [0]:

WITH customer_rev AS (
  SELECT customer_region, SUM(net_revenue) AS net_revenue
  FROM gold.fact_sales
  GROUP BY 1
),
supplier_rev AS (
  SELECT supplier_region, SUM(net_revenue) AS net_revenue
  FROM gold.fact_sales
  GROUP BY 1
)
SELECT
  COALESCE(c.customer_region, s.supplier_region)               AS region,
  ROUND(c.net_revenue, 2)                                      AS customer_revenue,
  ROUND(100.0 * c.net_revenue / SUM(c.net_revenue) OVER (), 2) AS customer_pct,
  ROUND(s.net_revenue, 2)                                      AS supplier_revenue,
  ROUND(100.0 * s.net_revenue / SUM(s.net_revenue) OVER (), 2) AS supplier_pct,
  ROUND(c.net_revenue - s.net_revenue, 2)                      AS revenue_gap
FROM customer_rev c
FULL OUTER JOIN supplier_rev s
        ON c.customer_region = s.supplier_region
ORDER BY region;

**Interpretation:** Customer and supplier footprints are nearly mirror images: **both are ~20% per region,** and the largest cross-region gap is only **±$1.9B (~0.17% of total revenue)** — suppliers in AMERICA ($218.87B) and ASIA ($219.08B) out-serve local customer demand, while EUROPE and MIDDLE EAST customers buy slightly more than local suppliers produce.

**Management action:** The footprint is effectively regionally self-sufficient — logistics should be optimized for intra-region cost, and only the small AMERICA/ASIA supply surplus is worth examining for cross-border freight or regional reallocation.